# Date Dimension Loader

This notebook generates and maintains the `warehouse.dim_date` dimension table.

**Purpose**: Generate a comprehensive date dimension covering historical and future dates

**Key Features**:
* Date keys in YYYYMMDD format for efficient joins
* Calendar attributes (year, quarter, month, week)
* Business day flags and fiscal period support
* Holiday calendar integration ready

**Target**: `workspace.warehouse.dim_date`

In [0]:
# Configuration
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Date range configuration
START_DATE = '2020-01-01'  # Historical start
END_DATE = '2030-12-31'    # Future end for planning

# Target table
TARGET_TABLE = 'workspace.warehouse.dim_date'

print(f"Generating date dimension from {START_DATE} to {END_DATE}")

In [0]:
# Generate date dimension
from pyspark.sql import Row
import pandas as pd

# Generate date range
start = datetime.strptime(START_DATE, '%Y-%m-%d')
end = datetime.strptime(END_DATE, '%Y-%m-%d')
date_range = pd.date_range(start, end, freq='D')

# Create date dimension with all attributes
date_data = []
for date in date_range:
    date_data.append(Row(
        date_sk=int(date.strftime('%Y%m%d')),
        date_value=date.date(),
        year=date.year,
        quarter=date.quarter,
        month=date.month,
        month_name=date.strftime('%B'),
        day_of_month=date.day,
        day_of_week=date.dayofweek + 1,  # 1=Monday, 7=Sunday
        day_of_week_name=date.strftime('%A'),
        day_of_year=date.dayofyear,
        week_of_year=date.isocalendar()[1],
        is_weekend=(date.dayofweek >= 5),
        is_month_start=(date.day == 1),
        is_month_end=(date.day == pd.Period(date, freq='D').days_in_month),
        is_quarter_start=(date.month % 3 == 1 and date.day == 1),
        is_quarter_end=(date.month % 3 == 0 and date.day == pd.Period(date, freq='D').days_in_month),
        is_year_start=(date.month == 1 and date.day == 1),
        is_year_end=(date.month == 12 and date.day == 31),
        fiscal_year=date.year if date.month >= 7 else date.year - 1,  # July start fiscal year
        fiscal_quarter=(date.quarter + 2) % 4 + 1 if date.month >= 7 else date.quarter + 2
    ))

# Create DataFrame
date_df = spark.createDataFrame(date_data)

print(f"Generated {date_df.count():,} date records")
date_df.display()

In [0]:
%sql
-- Create or replace date dimension table
CREATE OR REPLACE TABLE workspace.warehouse.dim_date (
  date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  date_value DATE NOT NULL COMMENT 'Actual date',
  year INT NOT NULL COMMENT 'Year',
  quarter INT NOT NULL COMMENT 'Quarter 1-4',
  month INT NOT NULL COMMENT 'Month 1-12',
  month_name STRING NOT NULL COMMENT 'Month name',
  day_of_month INT NOT NULL COMMENT 'Day of month',
  day_of_week INT NOT NULL COMMENT 'Day of week 1-7',
  day_of_week_name STRING NOT NULL COMMENT 'Day name',
  day_of_year INT NOT NULL COMMENT 'Day of year',
  week_of_year INT NOT NULL COMMENT 'ISO week number',
  is_weekend BOOLEAN NOT NULL COMMENT 'Is weekend day',
  is_month_start BOOLEAN NOT NULL COMMENT 'First day of month',
  is_month_end BOOLEAN NOT NULL COMMENT 'Last day of month',
  is_quarter_start BOOLEAN NOT NULL COMMENT 'First day of quarter',
  is_quarter_end BOOLEAN NOT NULL COMMENT 'Last day of quarter',
  is_year_start BOOLEAN NOT NULL COMMENT 'First day of year',
  is_year_end BOOLEAN NOT NULL COMMENT 'Last day of year',
  fiscal_year INT NOT NULL COMMENT 'Fiscal year (July start)',
  fiscal_quarter INT NOT NULL COMMENT 'Fiscal quarter',
  CONSTRAINT pk_dim_date PRIMARY KEY (date_sk)
)
COMMENT 'Date dimension for time-based analysis';

-- Insert generated data
INSERT OVERWRITE TABLE workspace.warehouse.dim_date
SELECT * FROM date_temp;

In [0]:
# Write to temp view then SQL will insert
date_df.createOrReplaceTempView('date_temp')
print("Date data ready for SQL insert")

In [0]:
%sql
-- Validate date dimension
SELECT 
  COUNT(*) as total_dates,
  MIN(date_value) as earliest_date,
  MAX(date_value) as latest_date,
  COUNT(DISTINCT year) as years,
  SUM(CASE WHEN is_weekend THEN 1 ELSE 0 END) as weekend_days,
  SUM(CASE WHEN NOT is_weekend THEN 1 ELSE 0 END) as weekdays
FROM workspace.warehouse.dim_date;